#Hands-on: Exploring LLM Behavior with Configurations

## Runtime and dependencies
Run this setup cell first in a fresh runtime, then run the remaining cells in order.
Public Hugging Face models/datasets in this lab do not require a token.
For Gemini, add **GOOGLE_API_KEY** in Colab Secrets and enable notebook access (`GEMINI_API_KEY` is also accepted). Locally, use an environment variable. Never paste a key into a cell. Gemini calls use your API quota.


In [ ]:
%pip install -q "openai>=1,<3"

## Introduction

In this hands-on, you’ll **experiment with generation parameters** to understand how they influence the behavior of a Large Language Model. You'll run controlled experiments to see how the model's tone, creativity, and coherence shift with different configurations

What You'll Learn:

* How to **tune generation parameters** like `temperature` and `top_p` to control randomness and diversity
* How to run **side-by-side comparisons** of model outputs across multiple settings
* How to apply this knowledge to **creative writing, technical answers, or branding consistency**
* How to structure simple experiments for **LLM behavior profiling**

This hands-on will give you practical insight into **how LLMs think** — and how to steer their thinking to fit your goals.

---

## 1. Setup
You can use:
* OpenAI (GPT-4, GPT-3.5)
* Anthropic (Claude)
* Google (Gemini)
* Local (via Ollama or LM Studio: Mistral, LLaMA, etc.)


In [ ]:
import os

def get_google_api_key():
    """Read credentials without writing them into notebook source or output."""
    key = os.environ.get("GOOGLE_API_KEY") or os.environ.get("GEMINI_API_KEY")
    if key:
        return key
    try:
        from google.colab import userdata
    except ImportError:
        raise RuntimeError("Set GOOGLE_API_KEY in your environment before running this cell.") from None
    for name in ("GOOGLE_API_KEY", "GEMINI_API_KEY"):
        try:
            return userdata.get(name)
        except userdata.SecretNotFoundError:
            continue
    raise RuntimeError("Add GOOGLE_API_KEY to Colab Secrets and enable notebook access.")

from openai import OpenAI
google_api_key = get_google_api_key()
client = OpenAI(api_key=google_api_key, base_url="https://generativelanguage.googleapis.com/v1beta/openai/")
model_name = os.environ.get("GEMINI_MODEL", "gemini-3.5-flash-lite")


## 2. Temperature and max_tokens

This experiment investigates how the temperature parameter influences the creativity and randomness of the model's responses.The max_tokens parameter is set to 512 (including any reasoning tokens) to limit the length of the responses but you can change as you'd like.

* **Temperature 0.1:** Produces more focused and less speculative output.
* **Temperature 0.5:** Generates moderately varied and imaginative descriptions.
* **Temperature 0.9:** Results in highly creative and diverse responses.

In [ ]:
# Prompt to test
prompt ="Describe what the inside of a superintelligent AI's dream might look like."

# Temperatures to test
temperatures = [0.1, 0.5, 0.9]

# Loop through temperatures and print results
for temp in temperatures:
    print(f"\n Temperature: {temp}")
    response = client.chat.completions.create(
        model=model_name,
        messages=[{"role": "user", "content": prompt}],
        temperature=temp,
        max_tokens=512,
        n=1,
        reasoning_effort="minimal",
    )
    print(response.choices[0].message.content.strip())

## 3.Top P
This section explores the effect of the top_p parameter, which controls the diversity of the output by sampling from the most probable tokens whose cumulative probability exceeds the top_p value. The temperature is fixed at 0.5 for this experiment.

* **top_p 0.3:** Leads to more constrained and less varied responses.
* **top_p 0.7:** Offers a balance between focus and diversity.
* **top_p 1.0:** Generates highly diverse and imaginative descriptions

In [ ]:
# Prompt to test
prompt = "Describe what the inside of a superintelligent AI's dream might look like."

# Temperatures and top_p values to test
temp = 0.5
top_ps = [0.3, 0.7, 1.0]

# Loop through temperature and top_p combinations
for top_p in top_ps:
        print(f"\n=== Temperature: {temp} | Top-p: {top_p} ===")
        response = client.chat.completions.create(
            model=model_name,
            messages=[{"role": "user", "content": prompt}],
            temperature=temp,
            top_p=top_p,
            max_tokens=512,
            n=1,
        reasoning_effort="minimal",
        )
        print(response.choices[0].message.content.strip())

## Conclusion:

* **Temperature** controls the randomness and risk-taking of the model.
* **Top-p** adjusts how much of the probability distribution is considered.
